# 10장 분류: 다음 날 강수 여부

이 노트북은 웹 교재 [10장 분류: 다음 날 강수 여부](https://pidocs.aifusionlab.org/ai-python/ch10/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/seoul_weather_2021_2025.csv
!ls *.csv

## 10.1 분류 문제 정의

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch10/10-1/)

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("seoul_weather_2021_2025.csv", parse_dates=["date"])
day = df["date"].dt.day_of_year
df["doy_sin"] = np.sin(2 * np.pi * day / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * day / 365.25)

In [ ]:
df["target_rain"] = (df["precip_mm"].shift(-1) >= 1.0).astype(int)
df["temp_range"] = df["temp_max"] - df["temp_min"]
data = df.iloc[:-1]                 # 마지막 날은 다음 날 자료가 없으므로 제외

feats = ["precip_mm", "humidity", "temp_range", "sunshine_h",
         "wind_max", "doy_sin", "doy_cos"]
train = data[data["date"] < "2025-01-01"]
test = data[data["date"] >= "2025-01-01"]
print(train["target_rain"].value_counts())
print(round(test["target_rain"].mean(), 3))

## 10.2 결정트리 분류기

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch10/10-2/)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

clf = DecisionTreeClassifier(max_depth=3, random_state=0)
clf.fit(train[feats], train["target_rain"])
print(export_text(clf, feature_names=feats, decimals=1))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 6))
plot_tree(clf, feature_names=feats, class_names=["0", "1"], filled=True, impurity=False, fontsize=8)
plt.show()

## 10.3 정확도와 혼동행렬

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch10/10-3/)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score

p = clf.predict(test[feats])
print("정확도", round(accuracy_score(test["target_rain"], p), 3))
print("기준선", round((test["target_rain"] == 0).mean(), 3))
print(confusion_matrix(test["target_rain"], p))
print("재현율", round(recall_score(test["target_rain"], p), 3))
print("정밀도", round(precision_score(test["target_rain"], p), 3))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(test["target_rain"], p, display_labels=["no rain", "rain"], cmap="Blues")
plt.show()

## 10.4 클래스 불균형과 트리 깊이

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch10/10-4/)

In [ ]:
clf_b = DecisionTreeClassifier(max_depth=3, random_state=0, class_weight="balanced")
clf_b.fit(train[feats], train["target_rain"])
pb = clf_b.predict(test[feats])
print("정확도", round(accuracy_score(test["target_rain"], pb), 3))
print(confusion_matrix(test["target_rain"], pb))
print("재현율", round(recall_score(test["target_rain"], pb), 3))
print("정밀도", round(precision_score(test["target_rain"], pb), 3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(test["target_rain"], pb, display_labels=["no rain", "rain"], cmap="Blues")
plt.show()

In [ ]:
for d in [1, 2, 3, 4, 6, 10, None]:
    c = DecisionTreeClassifier(max_depth=d, random_state=0, class_weight="balanced")
    c.fit(train[feats], train["target_rain"])
    tr = accuracy_score(train["target_rain"], c.predict(train[feats]))
    te = accuracy_score(test["target_rain"], c.predict(test[feats]))
    rc = recall_score(test["target_rain"], c.predict(test[feats]))
    print(f"깊이 {str(d):>4}  학습 정확도 {tr:.3f}  평가 정확도 {te:.3f}  평가 재현율 {rc:.3f}")

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]
tr_acc, te_acc = [], []
for d in depths:
    c = DecisionTreeClassifier(max_depth=d, random_state=0, class_weight="balanced")
    c.fit(train[feats], train["target_rain"])
    tr_acc.append(accuracy_score(train["target_rain"], c.predict(train[feats])))
    te_acc.append(accuracy_score(test["target_rain"], c.predict(test[feats])))

plt.figure(figsize=(7, 3.5))
plt.plot(depths, tr_acc, marker="o", label="train accuracy")
plt.plot(depths, te_acc, marker="o", label="test accuracy")
plt.axhline(0.703, color="gray", linestyle="--", label="baseline (always 0)")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()